In [ ]:
# %% [code]
# Y.Ding, 2025 Jun 4
# This notebook post-process KCWI DRP reduced cubes with KSkywizard in a 'headless mode', without calling the GUI.
# Please make sure you know how to use KSkywizard in GUI before running this notebook. 
# It is dangerous to run this notebook without knowing what KSkywizard does.
import importlib
from kskywizard import kskywizard
importlib.reload(kskywizard)
import os,json,re
import tkinter as tk
from glob import glob
from astropy.io import fits
import numpy as np
import sys
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__
# Hide the GUI window since we run in notebook (headless mode)
root = tk.Tk()
root.withdraw()
# Instantiate the viewer application.
# (Assumes KCWIViewerApp is already defined in the notebook environment.)
app = kskywizard.KCWIViewerApp(root)

###############################################################################
# Step 0: Set directories
###############################################################################
# Set the path to your updated standard star invsens file (updated_invsens.fits)
# Define your input and output directories (adjust as needed)
datatype="red"
otime="2024sep03"
base_in=f"/Volumes/WD_BLACK/KCWIdata/{otime}"
base_out=f"/Volumes/S990EVOP/KCWIdata/{otime}"
input_dir = base_in+"/redux"       # <<== CHANGE THIS: directory containing DRP files
output_dir = base_out+"/ksky"    # <<== CHANGE THIS: where cropped files/results will be saved
invsens_file = base_in+f"/ksky/g191b2b_{datatype}_invsens_updated.fits"  # <<== CHANGE THIS: with your updated invsens file

#Y.D.: There are lots of updates in the latest KcwiKit, you will need the exact same version to generate the logs
# The latest pull request I made on KcwiKit's main repo should work. You can also use the version in my repo

#================================================================================
# Step 1: Generate observing log with KcwiKit, e.g.
#$ kcwi_gen_log 2024jul07_red.log -f kr*.fits            

    
# Step 2: Generate a json table using observing log. 
#$ kcrm_group_frames 2024jul07_red.log kcrm_2024jul07_red.json -c -d -i
# Note: the same function also works for blue data, but you need to add '-b'
#$ kcrm_group_frames 2024jul07_blue.log kcrm_2024jul07_blue.json -c -d -i -b



# Step 3: Open the generated .json file, take a look at the data grouping to make sure they are accurate. 
#a) The first frame in each group will be taken as reference for continuum source masking.
#b) You may also want to remove standard star observations in the table
#c) If you have sky frames, you need to change the group name of each sky frame group to {object group name}_sky, so that the program will know which sky frame corresponds to which data group
# If there are shared sky frames for two data group, it is Okay to just copy the same sky group but change the name of the group.
obj_dict = json.load(open(base_in+f"/kcrm_{otime}_{datatype}.json", 'r')) 

# Step 4: Run the following code to prepare invsens curve for KSkywizard
#select the sky frames from the object dictionary
selected_keys = [k for k in obj_dict.keys() if "sky" in k]
app.structure_var.set('icubed') # <<== CHANGE THIS: if you want to directly process the flux calibrated 'icubes'
if not os.path.exists(invsens_file):
    raise FileNotFoundError(f"Updated invsens file not found: {invsens_file}")

# Simulate the GUI field update
app.std_entry.delete(0, tk.END)
app.std_entry.insert(0, invsens_file)

# Instead of opening a file dialog, load the file directly:
hdu = fits.open(invsens_file)
hdr = hdu[0].header
# Build the wavelength array as in the original code
wvl = (np.arange(hdr['NAXIS1']) + 1 - hdr['CRPIX1']) * hdr['CDELT1'] + hdr['CRVAL1']

# Fill in the standard star dictionary using the "updated" format
app.std = {}
app.std['wave'] = wvl
app.std['invsens_data'] = hdu[0].data[0]
app.std['invsens_model_drp'] = hdu[0].data[1]
app.std['counts'] = hdu[0].data[2]
app.std['name'] = hdr['OBJECT'].lower().replace(" ", "")
app.std['invsens_model'] = hdu[0].data[3]
app.std['flag'] = hdu[0].data[4]
if len(hdu[0].data) > 5:
    app.std['tellmodel'] = hdu[0].data[5]
else:
    app.std['tellmodel'] = None
app.std['frame'] = os.path.basename(invsens_file).replace('_invsens_updated.fits', '')
app.std['statenam'] = hdr.get('statenam', '')
app.std['bspline_bkpt'] = 150
app.std['bspline_polyorder'] = 3
app.insert_text(f"[INFO] Loading the {app.std_entry.get()}")
print("Updated standard star invsens curve loaded.")
print(app.output_text.get("1.0", tk.END), file=sys.__stdout__)



In [ ]:
# Example structure of the JSON file
obj_dict

{'LyC22-AGN_2223_0_M_BL4600_RL7150_300': ['kr240903_00093.fits',
  'kr240903_00094.fits',
  'kr240903_00095.fits'],
 'LyC22-AGN_2223_45_M_BL4600_RL7150_300': ['kr240903_00096.fits',
  'kr240903_00097.fits',
  'kr240903_00098.fits'],
 'LyC22-AGN_2223_90_M_BL4600_RL7150_300': ['kr240903_00099.fits',
  'kr240903_00100.fits',
  'kr240903_00101.fits'],
 'LyC22-AGN_2223_135_M_BL4600_RL7150_300': ['kr240903_00102.fits',
  'kr240903_00103.fits',
  'kr240903_00104.fits'],
 'LyC22-AGN_2223_180_M_BL4600_RL7150_300': ['kr240903_00105.fits',
  'kr240903_00106.fits',
  'kr240903_00107.fits'],
 'LyC22-AGN_2223_225_M_BL4600_RL7150_300': ['kr240903_00108.fits',
  'kr240903_00109.fits',
  'kr240903_00110.fits'],
 'LyC22-AGN_2223_270_M_BL4600_RL7150_300': ['kr240903_00111.fits',
  'kr240903_00112.fits',
  'kr240903_00113.fits'],
 'LyC22-AGN_2223_315_M_BL4600_RL7150_300': ['kr240903_00114.fits',
  'kr240903_00115.fits',
  'kr240903_00116.fits'],
 'DSF2237b-D1_0_M_BL4600_RL7150_300_0': ['kr240903_00117.fit

In [ ]:
###############################################################################
# Step 5: Load the DRP reduced data
###############################################################################
# You must first run it with process = False and save_data = True to generate the white-light images, use the image to rule out continuum sources, and then run it with save_data=False, process = True to do the ZAP processing.


save_data = False # <<== CHANGE THIS: whether you want to save the cropped data
process = True # <<== CHANGE THIS: whether you want to process the data with either ZAP or pypeit
skip_skymask = False# <<== CHANGE THIS: whether you want to skip the sky mask generation step
app.use_zap.set(True)# <<== CHANGE THIS: whether we are going to do ZAP
app.use_telluric.set(True) # <<== CHANGE THIS: whether we are going to do telluric correction
app.use_Ha_seg.set(False) # <<== CHANGE THIS: whether we are going to do extra sky segment around Ha (note, relatively accurate redshift must be provided)
app.use_multi_skyseg.set(True) # <<== CHANGE THIS: whether we are going to use multiple sky segments
app.data_wvl_range=[-1,-1] # <<== CHANGE THIS: if you want to set a specific wavelength range for the data. "-1" corresponds to WAVEGOOD

#z=[0.036, 0.03,0.0703, 0.0355, 0.1085, 0.024, 0.1139, 0.06377, 0.10168] # <-- Change to the redshift of your target if needed
#skyframe=42
app.base = input_dir
app.output = output_dir
for ind,obj in enumerate(list(obj_dict.keys())):
    if "sky" in obj:
        continue
    elif obj+"_sky" in selected_keys:
        sky_group=obj_dict[obj+"_sky"]
        match_sky = re.search(r'0*([1-9]\d*)\.fits$', sky_group[0])
        sky_frame = int(match_sky.group(1))
    else:
        sky_group=None
        sky_frame = -1
        app.insert_text(f"[Warning] no sky frame found for {obj}, using in-field sky!")
        #raise Exception("Error:no sky frame found!")
    for fn in obj_dict[obj]:
        app.insert_text(f"[INFO] Processing {obj}")
        match = re.search(r'0*([1-9]\d*)\.fits$', fn)
        science_frame = int(match.group(1))
        
        app.index = science_frame
        app.index2 = sky_frame

        # Determine the prefix (assumes the DRP file names start with 'k'); change logic if necessary.
        drp_files = glob(os.path.join(input_dir, "kr*.fits")) # <<== CHANGE THIS: if you want to reduce blue channel
        if not drp_files:
            raise Exception("No DRP reduced files found in the input directory.")
        app.prefix = os.path.basename(drp_files[0])[:8]
        app.index_entry.delete(0, tk.END)
        app.index_entry.insert(0, f'{app.prefix}_{app.index:05d}')

        app.index2_entry.delete(0, tk.END)
        app.index2_entry.insert(tk.END, f'{app.prefix}_{app.index2:05d}')
        if sky_frame>0:
            app.insert_text(f"[INFO] Set the sky frame for science frame {app.prefix}_{app.index:05d}: {app.prefix}_{app.index2:05d} ")
        else:
            app.insert_text(f"[INFO] Set the sky frame for science frame {app.prefix}_{app.index:05d}: {app.prefix}_{app.index:05d} ")
        print("DRP reduced data parameters set.")

        if save_data:
            # Load the DRP reduced (raw) data cube.
            app.load_data(datatype='raw')
            print("DRP reduced data cube loaded.")
            ###############################################################################
            # Step 6: Save the cropped data
            ###############################################################################
            # This will crop your loaded data to the assigned region and save it along with
            # a white-light image and (if applicable) a preliminary ZAP mask.
            app.save_cropped_data()
            print("Cropped data saved.")
        ###############################################################################
        # Step 7: Load the cropped data
        ###############################################################################
        # Now load the cropped data (i.e. data saved in the output directory).
        if process:
            redshift=0.03#z[ind//2]#
            if "sky" in obj:
                continue
            app.load_data(datatype='cropped')
            print("Cropped data loaded.")
            #shared_mask_frame = science_frame - ((science_frame) % 3)

            ###############################################################################
            # Step 8: Run ZAP processing
            ###############################################################################
            if sky_frame>0:
                shared_mask_frame=sky_frame
            else:
                match = re.search(r'0*([1-9]\d*)\.fits$', obj_dict[obj][0])
                shared_mask_frame=int(match.group(1))
            if skip_skymask:
                app.mindex = -1
            else:
                app.mindex = shared_mask_frame
                app.mask_entry.delete(0, tk.END)
                app.mask_entry.insert(0, f'{shared_mask_frame:05d}')
                print(f"Mask entry updated to shared mask frame: {shared_mask_frame:05d}", file=sys.__stdout__)
                app.update_zap_mask() # zap mask for merger project as we have sky frames


            # Set additional parameters as needed (e.g. redshift)

            app.redshift = redshift
            app.redshift_entry.delete(0, tk.END)
            app.redshift_entry.insert(tk.END, str(redshift))

            # Initialize ZAP (sets up sky segments etc.) and then run ZAP.
            app.run_zap_precondition()
            app.run_zap()
            print("ZAP processing completed.")
            # The application uses matplotlib to update the canvas.
            # If you wish, you can extract and display the final spectrum plot from the app.ax figure,
            # for example, by using plt.show() in a notebook cell.
            print(app.output_text.get("1.0", tk.END), file=sys.__stdout__)
            #plt.show()
        else:
            print("Skipping ZAP ...")
os.system('say "Program Finished"')

0

In [49]:
fn

'kr250503_00116.fits'